# Ambulance Mission Gantt Chart

Interactive timeline of every ambulance mission from a single-run output.

Each row is one ambulance. Each bar is a mission, split into two phases:
- **Transit to pickup** (light colour) — from dispatch to arrival at the origin facility
- **Delivery** (dark colour) — from departure with patients to arrival at the destination

Hover on any bar to see: origin facility, destination, planned T1 / T2 / T3 counts.

In [ ]:
import sys
from pathlib import Path

_root = Path(".").resolve()
while not (_root / "paths.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))
import paths

import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [ ]:
FOLDER_NAME = "FOLDER_NAME"
run_folder = paths.SINGLE_RUN_OUTPUT / FOLDER_NAME

te = pd.read_excel(run_folder / "transport_events.xlsx")

# Parse JSON list columns that were serialised as strings
for col in ["patient_ids", "patient_triages"]:
    if col in te.columns and te[col].dtype == object:
        te[col] = te[col].apply(
            lambda x: json.loads(x) if isinstance(x, str) and x.startswith("[") else x
        )

print(f"Loaded {len(te)} transport events for {te['amb_id'].nunique()} ambulances")

## Reconstruct missions from transport events

A **mission** is everything between a `mission_start` and `mission_end` event
for a given ambulance. Within each mission we identify legs (departure → arrival
pairs) and classify them:

- **Outbound leg** — ambulance is empty, travelling to pick up patients
- **Return leg** — ambulance has patients on board, delivering them

In [ ]:
def build_missions(te: pd.DataFrame) -> pd.DataFrame:
    """
    Group transport events into missions and extract Gantt bar data.

    Returns a DataFrame with one row per mission, containing:
      amb_id, dispatch_time, pickup_arrival_time, delivery_departure_time,
      delivery_arrival_time, origin, destination, plan_triages,
      n_t1, n_t2, n_t3
    """
    missions = []
    for amb_id, grp in te.sort_values("time").groupby("amb_id"):
        rows = grp.to_dict("records")
        mission_events = []
        in_mission = False

        for row in rows:
            evt = row["event"]
            if evt == "mission_start":
                in_mission = True
                mission_events = [row]
            elif evt == "mission_end" and in_mission:
                mission_events.append(row)
                missions.append(_parse_mission(amb_id, mission_events))
                in_mission = False
                mission_events = []
            elif in_mission:
                mission_events.append(row)

    return pd.DataFrame(missions)


def _parse_mission(amb_id: str, events: list) -> dict:
    """Extract timing and cargo info from a list of events for one mission."""
    dispatch_time = events[0]["time"]

    # Separate departures and arrivals
    deps = [e for e in events if e["event"] == "departure"]
    arrs = [e for e in events if e["event"] == "arrival"]

    # Plan triages come from mission_start
    plan_triages = events[0].get("patient_triages", [])
    if isinstance(plan_triages, str):
        plan_triages = json.loads(plan_triages)

    n_t1 = sum(1 for t in plan_triages if str(t).startswith("T1"))
    n_t2 = sum(1 for t in plan_triages if str(t).startswith("T2"))
    n_t3 = sum(1 for t in plan_triages if str(t).startswith("T3"))

    # Outbound leg: first departure/arrival (empty ambulance going to pickup)
    # Return leg: last departure/arrival (loaded ambulance delivering)
    pickup_arrival_time = dispatch_time   # fallback
    delivery_depart_time = dispatch_time
    delivery_arrival_time = dispatch_time
    origin = ""
    destination = ""

    if arrs:
        # First arrival = arrived at pickup facility
        pickup_arrival_time = arrs[0]["time"]
        origin = arrs[0].get("to_loc", "")
    if len(deps) >= 2:
        # Second departure = leaving pickup with patients
        delivery_depart_time = deps[1]["time"]
    elif len(deps) == 1 and not arrs:
        # Single leg only
        delivery_depart_time = deps[0]["time"]
    if len(arrs) >= 2:
        delivery_arrival_time = arrs[-1]["time"]
        destination = arrs[-1].get("to_loc", "")
    elif len(arrs) == 1 and len(deps) <= 1:
        # Only one leg total
        delivery_arrival_time = arrs[0]["time"]
        destination = arrs[0].get("to_loc", "")

    return {
        "amb_id": amb_id,
        "dispatch_time": dispatch_time,
        "pickup_arrival_time": pickup_arrival_time,
        "delivery_depart_time": delivery_depart_time,
        "delivery_arrival_time": delivery_arrival_time,
        "origin": origin,
        "destination": destination,
        "n_t1": n_t1,
        "n_t2": n_t2,
        "n_t3": n_t3,
    }


missions = build_missions(te)
print(f"Reconstructed {len(missions)} missions across {missions['amb_id'].nunique()} ambulances")

## CCP → R1 ambulance Gantt

In [ ]:
def plot_gantt(missions_df: pd.DataFrame, title: str):
    """
    Plot an interactive Plotly Gantt chart for a set of ambulance missions.

    Each mission produces two bars on the same ambulance row:
      1. Transit to pickup  (light)  — dispatch_time → pickup_arrival_time
      2. Delivery            (dark)   — delivery_depart_time → delivery_arrival_time
    """
    if missions_df.empty:
        print(f"{title}: no missions to plot")
        return

    LIGHT = "rgba(52, 152, 219, 0.45)"   # transit to pickup
    DARK  = "rgba(41, 128, 185, 1.0)"     # delivery

    # Sort ambulances for consistent y-axis order
    amb_order = sorted(missions_df["amb_id"].unique())
    amb_y = {a: i for i, a in enumerate(amb_order)}

    fig = go.Figure()

    for _, m in missions_df.iterrows():
        y = amb_y[m["amb_id"]]
        cargo_text = (f"T1: {m['n_t1']}  T2: {m['n_t2']}  T3: {m['n_t3']}")
        hover_base = (
            f"<b>{m['amb_id']}</b><br>"
            f"Origin: {m['origin']}<br>"
            f"Destination: {m['destination']}<br>"
            f"Plan: {cargo_text}"
        )

        # ── Transit to pickup ───────────────────────────────────
        t0, t1 = m["dispatch_time"], m["pickup_arrival_time"]
        if t1 > t0:
            fig.add_trace(go.Bar(
                x=[t1 - t0],
                y=[m["amb_id"]],
                base=[t0],
                orientation="h",
                marker_color=LIGHT,
                marker_line_width=0,
                hovertemplate=(
                    hover_base +
                    f"<br><br><b>Transit to pickup</b><br>"
                    f"Dispatch: {t0:.1f} min<br>"
                    f"Arrived at origin: {t1:.1f} min<br>"
                    f"Duration: {t1-t0:.1f} min"
                    "<extra></extra>"
                ),
                showlegend=False,
            ))

        # ── Delivery ────────────────────────────────────────────
        t2, t3 = m["delivery_depart_time"], m["delivery_arrival_time"]
        if t3 > t2:
            fig.add_trace(go.Bar(
                x=[t3 - t2],
                y=[m["amb_id"]],
                base=[t2],
                orientation="h",
                marker_color=DARK,
                marker_line_width=0,
                hovertemplate=(
                    hover_base +
                    f"<br><br><b>Delivery</b><br>"
                    f"Left origin: {t2:.1f} min<br>"
                    f"Arrived at dest: {t3:.1f} min<br>"
                    f"Duration: {t3-t2:.1f} min"
                    "<extra></extra>"
                ),
                showlegend=False,
            ))

    # ── Invisible legend entries ────────────────────────────────
    fig.add_trace(go.Bar(
        x=[0], y=[amb_order[0]], base=[0], orientation="h",
        marker_color=LIGHT, name="Transit to pickup", showlegend=True,
        hoverinfo="skip",
    ))
    fig.add_trace(go.Bar(
        x=[0], y=[amb_order[0]], base=[0], orientation="h",
        marker_color=DARK, name="Delivery", showlegend=True,
        hoverinfo="skip",
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Simulation time (minutes)",
        yaxis_title="Ambulance",
        barmode="overlay",
        height=max(400, 40 * len(amb_order)),
        yaxis=dict(
            categoryorder="array",
            categoryarray=list(reversed(amb_order)),
        ),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        hoverlabel=dict(bgcolor="white", font_size=12),
    )
    fig.show()


# ── Classify ambulances by C2 prefix ─────────────────────────────────────
# CCP→R1 ambulances start from an AXP (start_loc_id prefix "axp_")
# R1→R2  ambulances start from an R1  (start_loc_id prefix "r1_")
# We detect this from the first mission's origin pattern.

def classify_ambulances(missions_df):
    """Split missions into CCP→R1 and R1→R2 groups based on origin/destination patterns."""
    ccp_r1_ambs = set()
    r1_r2_ambs = set()

    for amb_id, grp in missions_df.groupby("amb_id"):
        origins = grp["origin"].str.lower()
        destinations = grp["destination"].str.lower()
        # If any origin is a CCP or destination is an R1, it's a CCP→R1 ambulance
        if origins.str.contains("ccp").any() or destinations.str.startswith("r1_").any():
            ccp_r1_ambs.add(amb_id)
        elif origins.str.startswith("r1_").any() or destinations.str.startswith("r2").any():
            r1_r2_ambs.add(amb_id)
        else:
            # Fallback: check the from_loc of the first transport event
            ccp_r1_ambs.add(amb_id)

    m_ccp = missions_df[missions_df["amb_id"].isin(ccp_r1_ambs)]
    m_r1  = missions_df[missions_df["amb_id"].isin(r1_r2_ambs)]
    return m_ccp, m_r1


m_ccp_r1, m_r1_r2 = classify_ambulances(missions)
print(f"CCP→R1: {len(m_ccp_r1)} missions ({m_ccp_r1['amb_id'].nunique()} ambulances)")
print(f"R1→R2:  {len(m_r1_r2)} missions ({m_r1_r2['amb_id'].nunique()} ambulances)")

In [ ]:
plot_gantt(m_ccp_r1, "CCP → R1 — Ambulance missions")

## R1 → R2 ambulance Gantt

In [ ]:
plot_gantt(m_r1_r2, "R1 → R2 — Ambulance missions")

## Summary statistics

In [ ]:
for label, m_df in [("CCP → R1", m_ccp_r1), ("R1 → R2", m_r1_r2)]:
    if m_df.empty:
        print(f"{label}: no missions\n")
        continue

    m_df = m_df.copy()
    m_df["transit_dur"] = m_df["pickup_arrival_time"] - m_df["dispatch_time"]
    m_df["delivery_dur"] = m_df["delivery_arrival_time"] - m_df["delivery_depart_time"]
    m_df["total_dur"] = m_df["delivery_arrival_time"] - m_df["dispatch_time"]
    m_df["n_patients"] = m_df["n_t1"] + m_df["n_t2"] + m_df["n_t3"]

    print(f"═══ {label} ═══")
    print(f"  Missions:           {len(m_df)}")
    print(f"  Active ambulances:  {m_df['amb_id'].nunique()}")
    print(f"  Patients planned:   {m_df['n_patients'].sum():.0f}  "
          f"(T1: {m_df['n_t1'].sum():.0f}  T2: {m_df['n_t2'].sum():.0f}  T3: {m_df['n_t3'].sum():.0f})")
    print(f"  Missions per amb:   median={m_df.groupby('amb_id').size().median():.0f}  "
          f"max={m_df.groupby('amb_id').size().max()}")
    print(f"  Transit to pickup:  median={m_df['transit_dur'].median():.1f} min  "
          f"mean={m_df['transit_dur'].mean():.1f} min")
    print(f"  Delivery:           median={m_df['delivery_dur'].median():.1f} min  "
          f"mean={m_df['delivery_dur'].mean():.1f} min")
    print(f"  Total mission:      median={m_df['total_dur'].median():.1f} min  "
          f"mean={m_df['total_dur'].mean():.1f} min")
    print()